# Designed to merge a preprocessed dataset with an existing dataset into one singlar one for Data cleaning, Model processing and Standardization

In [83]:
import pandas as pd

In [84]:
uncleaned_csv_folder = "./preprocessed_weather_data/"
final_csv_folder = "./Merged Location Csvs/"

locationData = pd.read_csv("locationdata.csv", encoding='latin1' )
locationData.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16 entries, 0 to 15
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   locationURL   16 non-null     object
 1   locationCode  16 non-null     object
dtypes: object(2)
memory usage: 388.0+ bytes


In [85]:
locationData.head(10)

,locationURL,locationCode
0,Port%20of%20Spain%2C%20Trinidad%2C%20Trinidad%...,Port-of-Spain
1,San%20Fernando%2C%20Trinidad%2C%20Trinidad%20a...,San-Fernando
2,Arima%2C%20Trinidad%2C%20Trinidad%20and%20Tobago,Arima
3,Sangre%20Grande%2C%20Trinidad%2C%20Trinidad%20...,Sangre-Grande
4,Tunapuna-Piarco%2C%20Trinidad%2C%20Trinidad%20...,Tunapuna-Piarco
5,Crown%20Point%2C%20Tobago%2C%20Trinidad%20and%...,Crown-Point
6,Scarborough%2C%20Tobago%2C%20Trinidad%20and%20...,Scarborough
7,Rio%20Claro%20-%20Mayaro%2C%20Trinidad%2C%20Tr...,Rio-Claro-Mayaro
8,Siparia%2C%20Trinidad%2C%20Trinidad%20and%20To...,Siparia
9,Chaguanas%2C%20Trinidad%2C%20Trinidad%20and%20...,Chaguanas


In [86]:
# Extract columns into lists
locationUrl = locationData["locationURL"].tolist()
locationCode = locationData["locationCode"].tolist()

In [87]:
print("Ensured location URLs and codes are loaded:")
print(f"📌Location URL: {locationUrl[0]}")
print(f"📌Location Code: {locationCode[0]}")

Ensured location URLs and codes are loaded:
📌Location URL: Port%20of%20Spain%2C%20Trinidad%2C%20Trinidad%20and%20Tobago
📌Location Code: Port-of-Spain


In [96]:
import os
import pandas as pd

# Global stats
processed_files = set()
files_skipped = set()

def process_weather_files(uncleaned_csv_folder, final_csv_folder):
    print("--------------------------------🧠 Beginning weather data processing...--------------------------------")

    # check for folder existence
    if not os.path.exists(uncleaned_csv_folder):
        print(f"❌ Error: Folder '{uncleaned_csv_folder}' does not exist.")
        return

    for file in os.listdir(uncleaned_csv_folder):

        if not file.endswith(".csv"):
            continue

        print("\n-----------------------Processing File-----------------------")
        print(f"Processing file: {file}")

        try:
            # Extract metadata from filename
            parts = file.split("_")
            location_code = parts[0]
            enddate = parts[-1].split(".")[0]

            print(f"Location code: {location_code}")
            print(f"End date: {enddate}")

            raw_path = os.path.join(uncleaned_csv_folder, file)
            raw_df = pd.read_csv(raw_path)

            # Find matching cleaned file
            cleaned_df = None
            for cleaned_file in os.listdir(final_csv_folder):
                if cleaned_file.endswith(".csv") and cleaned_file.startswith(location_code):
                    cleaned_path = os.path.join(final_csv_folder, cleaned_file)
                    cleaned_df = pd.read_csv(cleaned_path)
                    print(f"Found existing file: {cleaned_file}")
                    break

            # Ensure datetime matches BEFORE combining
            raw_df["datetime"] = pd.to_datetime(raw_df["datetime"], format="mixed", errors="coerce")

            if cleaned_df is not None:
                cleaned_df["datetime"] = pd.to_datetime(cleaned_df["datetime"], format="mixed", errors="coerce")

                # APPEND
                combined_df = pd.concat([cleaned_df, raw_df], ignore_index=True)
            else:
                print("⚠️ No existing file found to merge → creating new dataset")
                combined_df = raw_df

            # Clean the combined DataFrame
            combined_df = (
                combined_df
                .dropna(subset=["datetime"])
                .sort_values("datetime")
                .drop_duplicates(subset="datetime", keep="last")
                .reset_index(drop=True)
            )

            # Save ONE final file
            output_name = f"{location_code}_weather_data_{enddate}.csv"
            output_path = os.path.join(final_csv_folder, output_name)

            combined_df.to_csv(output_path, index=False)

            print(f"✅ Saved: {output_name}")
            print(f"   Rows: {len(combined_df)}")

            processed_files.add(file)

            if cleaned_path and os.path.exists(cleaned_path):

                try:
                    os.remove(cleaned_path)
                    print(f"🗑️ Deleted old file: {os.path.basename(cleaned_path)}")
                except Exception as e:
                    print(f"⚠️ Failed to delete old file: {e}")
        except Exception as e:
            print(f"❌ Error processing {file}: {e}")
            files_skipped.add(file)

    print(f"\n------------------------Finished processing data-----------------------")

# RUN
process_weather_files(uncleaned_csv_folder, final_csv_folder)

--------------------------------🧠 Beginning weather data processing...--------------------------------

-----------------------Processing File-----------------------
Processing file: Arima_weather_data_2025-04-27_2026-04-04.csv
Location code: Arima
End date: 2026-04-04
Found existing file: Arima_weather_data_2000_2025.csv
✅ Saved: Arima_weather_data_2026-04-04.csv
   Rows: 9525
🗑️ Deleted old file: Arima_weather_data_2000_2025.csv

-----------------------Processing File-----------------------
Processing file: Chaguanas_weather_data_2025-04-27_2026-04-04.csv
Location code: Chaguanas
End date: 2026-04-04
Found existing file: Chaguanas_weather_data_2000_2025.csv
✅ Saved: Chaguanas_weather_data_2026-04-04.csv
   Rows: 9525
🗑️ Deleted old file: Chaguanas_weather_data_2000_2025.csv

-----------------------Processing File-----------------------
Processing file: Couva-Tabaquite-Talparo_weather_data_2025-04-27_2026-04-04.csv
Location code: Couva-Tabaquite-Talparo
End date: 2026-04-04
Found exis

In [97]:
# 📊 Summary
print("\n==================== SUMMARY ====================")
print(f"Processed files: {len(processed_files)}")
print(f"Skipped files: {len(files_skipped)}")

if files_skipped:
    print("Skipped:")
    for f in files_skipped:
        print(f" - {f}")



==================== SUMMARY ====================
Processed files: 16
Skipped files: 0


In [99]:
report_path = os.path.join(final_csv_folder, "report.txt")

with open(report_path, "w") as report_file:
    report_file.write("--------------------- CSV Merge Report --------------------\n")
    report_file.write(f"Date: {pd.Timestamp.now()}\n\n")

    report_file.write("Summary:\n")
    report_file.write(f"Processed files: {len(processed_files)}\n")
    report_file.write(f"Skipped files: {len(files_skipped)}\n\n")

    report_file.write("Processed files:\n")
    for f in processed_files:
        report_file.write(f" - {f}\n")

    report_file.write("\nSkipped files:\n")
    for f in files_skipped:
        report_file.write(f" - {f}\n")

print(f"📄 Report saved to: {report_path}")

📄 Report saved to: ./Merged Location Csvs/report.txt
